# 근거 판정 프롬프트 개선: 기준선 vs 수정안 (개발 세트)

담당 C (조환호). 로직은 `scripts/compare_judge_format.py`에 있고, 이 노트북은 설치·실행·내려받기만 합니다.

- **기준선**: PR #21 병합 직후 `main`(`c7fe41c`)의 근거 판정 프롬프트
- **수정안**: `HwanHo` 브랜치의 근거 판정 프롬프트
- 두 실행 모두 같은 설정(`configs/default.yaml`: k=5, m=7)과 같은 공통 문구, JSON 형식, 개발 세트 검색형 40문항

**시작 전:** `런타임 → 런타임 유형 변경 → T4 GPU`를 고르고 저장하세요. 셀을 위에서부터 하나씩 실행합니다.
세션이 끊기면(런타임이 초기화되지 않았다면) 2·4번 셀을 다시 실행한 뒤 끊긴 단계의 셀을 다시 실행하면 이어서 돕니다.

## 1. GPU 확인 (T4가 보여야 합니다)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. 코드 받기

In [ ]:
import os
REPO = "/content/Intelligence-Software-Project"
if not os.path.exists(REPO):
    !git clone https://github.com/minyi-k03/Intelligence-Software-Project.git {REPO}
%cd {REPO}
!git fetch -q origin
!git checkout -q HwanHo && git pull -q && git log --oneline -1

## 3. 의존성 설치

In [ ]:
!pip install -q -r requirements.txt

## 4. Ollama 설치·실행, 모델 받기 (모델 태그는 `configs/default.yaml`에서 읽음)

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, yaml
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
MODEL_TAG = yaml.safe_load(open("configs/default.yaml", encoding="utf-8"))["model"]["tag"]
print("모델:", MODEL_TAG)
!ollama pull {MODEL_TAG}
!ollama list

## 5. 동작 확인 (수정안으로 2문항, git에서 제외되는 임시 폴더)

In [ ]:
!python -m scripts.compare_judge_format --formats json --limit 2 --out-dir results/runs/_scratch/c_judge_prompt_check

## 6. 기준선 실행 (`c7fe41c`, 40문항). 끊기면 이 셀을 다시 실행

In [ ]:
!git checkout -q c7fe41c && git log --oneline -1
!python -m scripts.compare_judge_format --formats json --out-dir results/runs/dev/c_judge_prompt/baseline

## 7. 수정안 실행 (`HwanHo`, 40문항). 끊기면 이 셀을 다시 실행

In [ ]:
!git checkout -q HwanHo && git log --oneline -1
!python -m scripts.compare_judge_format --formats json --out-dir results/runs/dev/c_judge_prompt/revised

## 8. 비교 표

In [ ]:
!python -m scripts.compare_judge_format --formats json --out-dir results/runs/dev/c_judge_prompt --compare baseline=results/runs/dev/c_judge_prompt/baseline revised=results/runs/dev/c_judge_prompt/revised

## 9. 결과 내려받기

내려받은 zip은 다운로드 폴더에 그대로 두시면 됩니다.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/c_judge_prompt", "zip", "results/runs/dev/c_judge_prompt")
files.download("/content/c_judge_prompt.zip")